# Chicago taxi trips — week of 11–17 September 2023

**Summary (filled in at the end of the analysis)**

SUMMARY_PLACEHOLDER

## 1. Data loading

All figures describe **trips reported to the City of Chicago**, not total market volume (per `dataset.md`, not every trip is reported).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('figures/analysis', exist_ok=True)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Chart style: one blue for the main series, orange only for a deliberate highlight, recessive grid
BLUE, ORANGE, GREY, INK = '#2a78d6', '#eb6834', '#b9b8b3', '#0b0b0b'
plt.rcParams.update({
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': '#e6e5e1', 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'figure.dpi': 100, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.titlelocation': 'left', 'axes.labelcolor': '#52514e', 'xtick.color': '#52514e', 'ytick.color': '#52514e',
})

raw = pd.read_csv(
    'trips.csv',
    dtype={'pickup_census_tract': 'string', 'dropoff_census_tract': 'string'},
    parse_dates=['trip_start_timestamp', 'trip_end_timestamp'],
)
print('shape:', raw.shape)
print('trip_id unique:', raw.trip_id.is_unique)
print('start range:', raw.trip_start_timestamp.min(), '->', raw.trip_start_timestamp.max())

## 2. Data checks and cleaning

### 2.1 Missing values

In [ ]:
miss = raw.isna().sum()
miss = miss[miss > 0].to_frame('missing')
miss['pct_of_rows'] = (miss.missing / len(raw) * 100).round(2)
miss

Notes: the five money columns (`fare`, `tips`, `tolls`, `extras`, `trip_total`) are all missing together on the same rows. The census-tract and dropoff columns are not used in this analysis (the brief says to use community areas).

### 2.2 Implausible or inconsistent values

In [ ]:
money = ['fare', 'tips', 'tolls', 'extras', 'trip_total']
has_money = raw.trip_total.notna()
parts_gap = (raw.trip_total - raw[['fare', 'tips', 'tolls', 'extras']].sum(axis=1)).round(2)
speed_mph = raw.trip_miles / (raw.trip_seconds / 3600)

checks = pd.DataFrame([
    ('Money fields missing (all five together)',          (~has_money).sum()),
    ('fare = 0 or trip_total = 0 (money present)',        (has_money & ((raw.fare <= 0) | (raw.trip_total <= 0))).sum()),
    ('trip_total > $500',                                 (raw.trip_total > 500).sum()),
    ('trip_total > sum of fare+tips+tolls+extras',        (parts_gap > 0.01).sum()),
    ('trip_total < sum of parts',                         (parts_gap < -0.01).sum()),
    ('   ... largest gap above the sum of parts ($)',     parts_gap.max()),
    ('trip_miles = 0',                                    (raw.trip_miles == 0).sum()),
    ('trip_miles > 100',                                  (raw.trip_miles > 100).sum()),
    ('trip_seconds = 0',                                  (raw.trip_seconds == 0).sum()),
    ('trip_seconds missing',                              raw.trip_seconds.isna().sum()),
    ('trip_seconds > 6 hours',                            (raw.trip_seconds > 6 * 3600).sum()),
    ('Implied speed > 100 mph (seconds > 0)',             (speed_mph > 100).sum()),
    ('extras > $100',                                     (raw.extras > 100).sum()),
    ('Duplicate rows apart from trip_id',                 raw.duplicated([c for c in raw.columns if c != 'trip_id']).sum()),
], columns=['check', 'rows']).set_index('check')
checks['rows'] = [f'${v:,.2f}' if 'largest gap' in i else f'{int(v):,}' for i, v in checks['rows'].items()]
checks